# NB08 — Quantization & Dequant Kernels

**North star callback:** QLoRA keeps the base model in 4-bit NF4 to cut VRAM by ~4×,
enabling 8B-parameter fine-tuning on a single 24 GB GPU. The catch: every forward pass
must dequantize NF4 weights back to bf16 before the matmul. This notebook builds NF4
from scratch and writes a Triton dequant kernel to understand what Unsloth optimizes.

## 1. NF4 quantization math

In [1]:
import torch

# 16 quantiles of N(0,1) mapped to [-1, 1] — the NF4 lookup table (matches bitsandbytes).
NF4_TABLE = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
    -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
     0.07958029955625534,  0.16093020141124725,  0.24611230194568634,  0.33791524171829224,
     0.44070982933044434,  0.5626170039176941,   0.7229568362236023,   1.0,
], dtype=torch.float32)

BLOCK_SIZE = 64  # elements per absmax block (bitsandbytes default)


def quantize_nf4(w: torch.Tensor):
    """Quantize a float32 tensor to NF4. Returns (packed_uint8, absmax, n_orig)."""
    w_flat = w.float().flatten()
    n = w_flat.numel()

    pad = (BLOCK_SIZE - n % BLOCK_SIZE) % BLOCK_SIZE
    if pad:
        w_flat = torch.cat([w_flat, torch.zeros(pad)])

    n_blocks = w_flat.numel() // BLOCK_SIZE
    blocks   = w_flat.view(n_blocks, BLOCK_SIZE)

    absmax     = blocks.abs().max(dim=1).values               # (n_blocks,)
    normalized = blocks / absmax.unsqueeze(1).clamp(min=1e-8) # [-1, 1]

    dists   = (normalized.flatten().unsqueeze(1) - NF4_TABLE.unsqueeze(0)).abs()
    indices = dists.argmin(dim=1).to(torch.uint8)              # [0, 15] per element

    # Pack: low nibble = even-indexed element, high nibble = odd-indexed element
    packed = (indices[0::2] & 0x0F) | ((indices[1::2] & 0x0F) << 4)
    return packed, absmax, n


def dequantize_nf4(packed: torch.Tensor, absmax: torch.Tensor, n: int) -> torch.Tensor:
    """Dequantize NF4 back to float32."""
    low  = (packed & 0x0F).long()
    high = ((packed >> 4) & 0x0F).long()
    indices = torch.stack([low, high], dim=1).flatten()

    values   = NF4_TABLE[indices]
    n_blocks = absmax.shape[0]
    values   = values[:n_blocks * BLOCK_SIZE].view(n_blocks, BLOCK_SIZE)
    values   = (values * absmax.unsqueeze(1)).flatten()
    return values[:n]


torch.manual_seed(42)
w = torch.randn(4096, 4096)  # typical attention projection weight

packed, absmax, n = quantize_nf4(w)
w_deq = dequantize_nf4(packed, absmax, n).view_as(w)

bf16_bytes = w.numel() * 2
nf4_bytes  = packed.numel() + absmax.numel() * 4

print(f'Weight : {w.shape}  ({w.numel():,} elements)')
print(f'bf16   : {bf16_bytes / 1024**2:.1f} MB')
print(f'NF4    : {nf4_bytes  / 1024**2:.1f} MB  '
      f'({packed.numel()/1024**2:.1f} MB packed + {absmax.numel()*4/1024**2:.1f} MB absmax)')
print(f'Ratio  : {bf16_bytes / nf4_bytes:.2f}x')
print()
print(f'Max quant error  : {(w - w_deq).abs().max().item():.4f}')
print(f'Mean quant error : {(w - w_deq).abs().mean().item():.6f}')

Weight : torch.Size([4096, 4096])  (16,777,216 elements)
bf16   : 32.0 MB
NF4    : 9.0 MB  (8.0 MB packed + 1.0 MB absmax)
Ratio  : 3.56x

Max quant error  : 0.6283
Mean quant error : 0.072779


## 2. Memory savings for Llama 3 8B

In [2]:
# Llama 3 8B architecture
D, I, KVH, VH, L = 4096, 14336, 8, 128, 32

per_layer = {
    'q_proj'   : D * D,
    'k_proj'   : D * (KVH * VH),
    'v_proj'   : D * (KVH * VH),
    'o_proj'   : D * D,
    'gate_proj': D * I,
    'up_proj'  : D * I,
    'down_proj': I * D,
}
total_params = sum(per_layer.values()) * L

bf16_gb       = total_params * 2   / 1024**3
nf4_data_gb   = total_params * 0.5 / 1024**3
nf4_absmax_gb = (total_params // BLOCK_SIZE) * 4 / 1024**3
nf4_total_gb  = nf4_data_gb + nf4_absmax_gb

print(f'Main weight params : {total_params/1e9:.2f}B')
print(f'bf16 VRAM          : {bf16_gb:.2f} GB')
print(f'NF4 VRAM           : {nf4_total_gb:.2f} GB  '
      f'(data {nf4_data_gb:.2f} + absmax {nf4_absmax_gb:.3f})')
print(f'Reduction          : {bf16_gb / nf4_total_gb:.1f}x  '
      f'({bf16_gb - nf4_total_gb:.2f} GB freed)')
print()
# Double quantization: quantize the absmax values themselves (fp32 -> int8 + second absmax)
# bitsandbytes compress_statistics=True does this with BLOCK2=256 for the second level.
BLOCK2 = 256
n_blocks1    = total_params // BLOCK_SIZE
dq_int8_gb   = n_blocks1 / 1024**3                       # int8: 1 byte per absmax
dq_absmax_gb = (n_blocks1 // BLOCK2) * 4 / 1024**3       # float32: 1 per 256 absmax
print(f'Double quantization absmax: '
      f'{nf4_absmax_gb*1024:.0f} MB -> {(dq_int8_gb+dq_absmax_gb)*1024:.0f} MB  '
      f'({nf4_absmax_gb/(dq_int8_gb+dq_absmax_gb):.1f}x reduction in overhead)')

Main weight params : 6.98B
bf16 VRAM          : 13.00 GB
NF4 VRAM           : 3.66 GB  (data 3.25 + absmax 0.406)
Reduction          : 3.6x  (9.34 GB freed)

Double quantization absmax: 416 MB -> 106 MB  (3.9x reduction in overhead)


## 3. bitsandbytes in practice

In [3]:
import bitsandbytes.functional as F

torch.manual_seed(0)
w_gpu = torch.randn(4096, 4096, device='cuda', dtype=torch.float16)

# quantize_4bit returns (quantized_uint8, QuantState)
w_q, quant_state = F.quantize_4bit(
    w_gpu.contiguous(), blocksize=64, quant_type='nf4', compress_statistics=False,
)

w_deq_bnb = F.dequantize_4bit(w_q, quant_state=quant_state, quant_type='nf4')

print(f'Original   : {w_gpu.dtype}  {tuple(w_gpu.shape)}  {w_gpu.numel()*2/1024**2:.0f} MB')
print(f'Quantized  : {w_q.dtype}  {tuple(w_q.shape)}  {w_q.numel()/1024**2:.1f} MB')
print(f'Absmax     : shape={quant_state.absmax.shape}  dtype={quant_state.absmax.dtype}')
print(f'Dequantized: {w_deq_bnb.dtype}  {tuple(w_deq_bnb.shape)}')
print()
err = (w_gpu.float() - w_deq_bnb.float()).abs()
print(f'Max error  : {err.max().item():.4f}')
print(f'Mean error : {err.mean().item():.6f}')

Original   : torch.float16  (4096, 4096)  32 MB
Quantized  : torch.uint8  (8388608, 1)  8.0 MB
Absmax     : shape=torch.Size([262144])  dtype=torch.float32
Dequantized: torch.float16  (4096, 4096)

Max error  : 0.5938
Mean error : 0.072774


## 4. Triton dequant kernel

Each program handles `CHUNK` elements (= `CHUNK//2` packed bytes).
It unpacks low and high nibbles, looks up the NF4 table (16 floats — fits in L1 cache),
then scales by the per-block absmax and stores as bfloat16.

In [4]:
import sys
sys.path.insert(0, '..')
from utils.benchmark import compare

import triton
import triton.language as tl


@triton.jit
def nf4_dequant_kernel(
    Packed_ptr,            # uint8, (N//2,) — two 4-bit indices per byte
    Absmax_ptr,            # float32, (N//BLOCK_SIZE,) — per-block scale
    Table_ptr,             # float32, (16,) — NF4 lookup table
    Out_ptr,               # bfloat16, (N,) — output
    N,
    BLOCK_SIZE: tl.constexpr,  # 64
    CHUNK: tl.constexpr,       # elements per program (multiple of 2)
):
    pid         = tl.program_id(0)
    chunk_start = pid * CHUNK

    byte_offs = tl.arange(0, CHUNK // 2)
    byte_addr = chunk_start // 2 + byte_offs
    mask_b    = byte_addr < (N + 1) // 2

    raw      = tl.load(Packed_ptr + byte_addr, mask=mask_b, other=0).to(tl.int32)
    idx_even = raw & 0xF
    idx_odd  = (raw >> 4) & 0xF

    val_even = tl.load(Table_ptr + idx_even)  # gather from 16-entry table (cached in L1)
    val_odd  = tl.load(Table_ptr + idx_odd)

    elem_even = chunk_start + byte_offs * 2
    elem_odd  = chunk_start + byte_offs * 2 + 1
    mask_e    = elem_even < N
    mask_o    = elem_odd  < N

    abs_even = tl.load(Absmax_ptr + elem_even // BLOCK_SIZE, mask=mask_e, other=1.0)
    abs_odd  = tl.load(Absmax_ptr + elem_odd  // BLOCK_SIZE, mask=mask_o, other=1.0)

    tl.store(Out_ptr + elem_even, (val_even * abs_even).to(tl.bfloat16), mask=mask_e)
    tl.store(Out_ptr + elem_odd,  (val_odd  * abs_odd ).to(tl.bfloat16), mask=mask_o)


def triton_dequant_nf4(
    packed: torch.Tensor,
    absmax: torch.Tensor,
    table: torch.Tensor,
    n: int,
) -> torch.Tensor:
    out   = torch.empty(n, device=packed.device, dtype=torch.bfloat16)
    CHUNK = 128
    nf4_dequant_kernel[triton.cdiv(n, CHUNK),](
        packed, absmax, table, out,
        N=n, BLOCK_SIZE=64, CHUNK=CHUNK,
    )
    return out


# --- Correctness check ---
torch.manual_seed(42)
w_cpu = torch.randn(4096, 4096)
packed_cpu, absmax_cpu, n_orig = quantize_nf4(w_cpu)

packed_gpu = packed_cpu.cuda()
absmax_gpu = absmax_cpu.cuda()
table_gpu  = NF4_TABLE.cuda()

# Both paths: float32 multiply -> round to bfloat16; results must be bit-identical
ref = dequantize_nf4(packed_cpu, absmax_cpu, n_orig).bfloat16()
out = triton_dequant_nf4(packed_gpu, absmax_gpu, table_gpu, n_orig).cpu()

max_err = (ref.float() - out.float()).abs().max().item()
print(f'Max error (Triton vs Python reference): {max_err}  ', end='')
print('✓' if max_err == 0.0 else f'✗  ({max_err:.6f} — check packing order)')

Max error (Triton vs Python reference): 0.0  ✓


## 5. Benchmark

In [5]:
w_bench = torch.randn(4096, 4096, device='cuda', dtype=torch.float16).contiguous()
w_q_bench, qs_bench = F.quantize_4bit(
    w_bench, blocksize=64, quant_type='nf4', compress_statistics=False,
)

results = compare(
    fns={
        'bnb_dequant'   : lambda: F.dequantize_4bit(w_q_bench, quant_state=qs_bench,
                                                     quant_type='nf4'),
        'triton_dequant': lambda: triton_dequant_nf4(packed_gpu, absmax_gpu,
                                                     table_gpu, n_orig),
    },
    notebook='nb08',
    experiment='nf4_dequant',
    n_warmup=10, n_repeat=50,
)
for label, r in results.items():
    print(f'{label:16s}: {r.latency_ms:.3f} ms')
print()
print('Note: Unsloth fuses dequant into the matmul — no separate bfloat16 weight tensor')
print('is written to HBM at all. That is the real speedup (NB09 covers this).')

bnb_dequant     : 0.051 ms
triton_dequant  : 0.077 ms

Note: Unsloth fuses dequant into the matmul — no separate bfloat16 weight tensor
is written to HBM at all. That is the real speedup (NB09 covers this).


## 6. Read Unsloth's dequant implementation

`fast_dequantize` in `unsloth/kernels/utils.py` is called inside `matmul_lora`
(seen in NB07). It dequantizes the base weight immediately before the matmul,
keeping the bfloat16 intermediate in registers rather than writing it back to HBM.

In [6]:
import inspect
sys.path.insert(0, '../unsloth')
from unsloth.kernels.utils import fast_dequantize
print(inspect.getsource(fast_dequantize))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
    @torch.inference_mode
    def fast_dequantize(W, quant_state = None, out = None, use_global_buffer = False):
        if isinstance(W, Float8Tensor):
            return W.dequantize()
        if quant_state is None:
            return W
        if W.dtype == torch.float8_e4m3fn:
            return weight_dequant(W, quant_state)
        if type(quant_state) is not list:
            # New quant_state as a class
            # https://github.com/TimDettmers/bitsandbytes/pull/763/files
            absmax = quant_state.absmax
            shape = quant_state.shape
            dtype = quant_state.dtype
            blocksize = quant_state.blocksize
            offset = quant_state.offset
            state2 = quant_state.state2
     

## 7. Exercises

1. **FP4 variant**: NF4 uses normal-distribution quantiles; FP4 uses evenly-spaced levels.
   Add an `fp4=True` flag to `quantize_nf4` / `dequantize_nf4` using bitsandbytes'
   FP4 table. How does quantization error compare for transformer weight distributions?

2. **Fused dequant + matmul**: modify `triton_dequant_nf4` to also accept an activation
   `X` of shape `(M, K)` and compute `X @ W_deq` without writing the dequantized weight
   to HBM. This is the key insight behind Unsloth's `matmul_lora` / `fast_dequantize`.

3. **Double quantization**: implement `quantize_nf4` with `compress_statistics=True`
   (matching bitsandbytes). Quantize the float32 absmax values to int8 with a second
   block size of 256. Verify error increases only slightly while absmax overhead drops ~4×.